In [0]:
%sql
-- Listar unidade de medida com quantidade e percentual do total
WITH totals AS (
  SELECT COUNT(*) as total_count
  FROM silver.cno
)
SELECT 
  unidade_de_medida,
  COUNT(*) as quantidade,
  ROUND(COUNT(*) * 100.0 / totals.total_count, 2) as percentual
FROM silver.cno
CROSS JOIN totals
GROUP BY unidade_de_medida, totals.total_count
ORDER BY quantidade DESC;

Na lista de unidades de medida, não foram encontradas ocorrências válidas diferentes de `m²`.

As  ocorrências inconsistentes com possibilidade de correção foram:
- `km`: possivelmente deveria ser `km²`;
- `,m2`: possivelmente decorrente de um erro de digitação.

Não será realizada nenhuma correção automática dessas ocorrências, considerando a quantidade de registros e a ausência de evidências suficientes para determinar a unidade correta com segurança.

In [0]:
%sql
-- Quantidade de registros e percentual por faixa de area_total
WITH totals AS (
  SELECT COUNT(*) as total_count
  FROM silver.cno
),
ranges AS (
  SELECT 
    CASE 
      WHEN area_total <= 100 THEN '0 - 100'
      WHEN area_total > 100 AND area_total <= 1000 THEN '100 - 1.000'
      WHEN area_total > 1000 AND area_total <= 10000 THEN '1.000 - 10.000'
      WHEN area_total > 10000 AND area_total <= 100000 THEN '10.000 - 100.000'
      WHEN area_total > 100000 AND area_total <= 500000 THEN '100.000 - 500.000'
      WHEN area_total > 500000 THEN '> 500.000'
      ELSE 'NULL ou inválido'
    END as faixa_area,
    CASE 
      WHEN area_total <= 100 THEN 1
      WHEN area_total > 100 AND area_total <= 1000 THEN 2
      WHEN area_total > 1000 AND area_total <= 10000 THEN 3
      WHEN area_total > 10000 AND area_total <= 100000 THEN 4
      WHEN area_total > 100000 AND area_total <= 500000 THEN 5
      WHEN area_total > 500000 THEN 6
      ELSE 7
    END as ordem
  FROM silver.cno
)
SELECT 
  r.faixa_area,
  COUNT(*) as quantidade,
  ROUND(COUNT(*) * 100.0 / t.total_count, 2) as percentual
FROM ranges r
CROSS JOIN totals t
GROUP BY r.faixa_area, r.ordem, t.total_count
ORDER BY r.ordem;


In [0]:
%sql
SELECT
  COUNT(*) AS total,

  COUNT(CASE WHEN metragem < 10 THEN 1 END) AS menor_10,
  ROUND(COUNT(CASE WHEN metragem < 10 THEN 1 END) * 100.0 / COUNT(*), 2) AS perc_menor_10,

  COUNT(CASE WHEN metragem > 10000 THEN 1 END) AS maior_10000,
  ROUND(COUNT(CASE WHEN metragem > 50000 THEN 1 END) * 100.0 / COUNT(*), 2) AS perc_maior_10000

FROM silver.cno_areas
WHERE tipo_de_area = 'Principal'
  AND categoria IN ('Existente', 'Obra Nova')
  AND destinacao = 'Residencial unifamiliar';



#### Análise da área total

A análise da distribuição da `area_total` não permitiu identificar um limite superior suficientemente confiável para a remoção de registros.

Embora valores acima de `500.000 m²` sejam pouco frequentes (689 registros, 0,02%), a tabela `CNO` contempla uma grande variedade de tipos de construção, incluindo **conjuntos habitacionais, edifícios de garagens e outros empreendimentos de grande porte**. Dessa forma, não é possível determinar, apenas com base na área total, se esses registros representam valores inconsistentes ou construções legítimas de grande dimensão.

Consequentemente, **nenhum registro foi removido com base exclusivamente no limite superior da `area_total`**. Os valores extremos serão mantidos para análises posteriores e poderão ser avaliados em conjunto com outras informações, como o tipo de construção e as áreas detalhadas disponíveis na tabela `CNO_AREAS`.

In [0]:
%sql
SELECT
  CASE
    WHEN metragem > 0 AND metragem <= 10 THEN '0 - 10'
    WHEN metragem > 10 AND metragem <= 30 THEN '10 - 30'
    WHEN metragem > 30 AND metragem <= 70 THEN '30 - 70'
    WHEN metragem > 70 AND metragem <= 100 THEN '70 - 100'
    WHEN metragem > 100 AND metragem <= 500 THEN '100 - 500'
    WHEN metragem > 500 AND metragem <= 1000 THEN '500 - 1.000'
    WHEN metragem > 1000 AND metragem <= 5000 THEN '1.000 - 5.000'
    WHEN metragem > 5000 AND metragem <= 10000 THEN '5.000 - 10.000'
    WHEN metragem > 10000 AND metragem <= 50000 THEN '10.000 - 50.000'
    WHEN metragem > 50000 THEN '> 50.000'
    ELSE 'Inválido ou NULL'
  END AS faixa_metragem,
  destinacao,
  COUNT(*) AS quantidade
FROM silver.cno_areas
WHERE tipo_de_area = 'Principal'
  AND categoria IN ('Existente', 'Obra Nova')
  AND destinacao IN ('Residencial unifamiliar', 'Casa popular')
GROUP BY faixa_metragem, destinacao
ORDER BY 
  CASE
    WHEN faixa_metragem = '0 - 10' THEN 1
    WHEN faixa_metragem = '10 - 30' THEN 2
    WHEN faixa_metragem = '30 - 70' THEN 3
    WHEN faixa_metragem = '70 - 100' THEN 4
    WHEN faixa_metragem = '100 - 500' THEN 5
    WHEN faixa_metragem = '500 - 1.000' THEN 6
    WHEN faixa_metragem = '1.000 - 5.000' THEN 7
    WHEN faixa_metragem = '5.000 - 10.000' THEN 8
    WHEN faixa_metragem = '10.000 - 50.000' THEN 9
    WHEN faixa_metragem = '> 50.000' THEN 10
    ELSE 11
  END, destinacao

Databricks visualization. Run in Databricks to view.

### Avaliação da distribuição da metragem

A maior concentração dos registros classificados como `Residencial unifamiliar` ou `Casa popular`
está na faixa de **100 a 500 m²**.

A baixa representatividade de construções com metragem inferior a 70 m² pode
estar relacionada aos critérios de obrigatoriedade de registro no CNO. A
legislação prevê situações específicas de dispensa, incluindo determinadas
construções residenciais unifamiliares de até **70 m²**, quando atendidos
requisitos específicos, como a condição do proprietário e a ausência de mão de
obra remunerada.

Esse critério pode resultar em uma **sub-representação de pequenas
construções na base**, fazendo com que a distribuição observada não represente
necessariamente a distribuição real das construções residenciais unifamiliares.